# Reporting & Dashboards with UAEF

## Scenario

You've already evaluated your QA agent across three frameworks (Bedrock, LangGraph, LangChain).
Now stakeholders need:

1. An **Executive Dashboard** — high-level performance overview
2. A **Comparison Dashboard** — side-by-side framework comparison
3. A **Regression Dashboard** — what broke between versions
4. A **Safety Dashboard** — responsible AI metrics
5. A **Dimension Deep-Dive** — drill into a specific evaluation dimension
6. **Export** everything to JSON, HTML, and CSV

This notebook covers every reporting and dashboard capability in UAEF, using the real
batch evaluation CSVs from `output/evaluation-results/`.

---
## 1. Setup & Imports

In [ ]:
import sys, os
import pandas as pd
from uuid import uuid4

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

# Experiment management (to create runs for comparison/regression dashboards)
from uaef.experiments import (
    ExperimentManager, AgentDesign, ComparisonEngine, RegressionDetector,
)

# Reporting
from uaef.reporting import (
    ExecutiveDashboardGenerator,
    ComparisonDashboardGenerator,
    ReportExporter,
    VisualizationEngine,
    Chart, ChartType, Dashboard, Report, ReportFormat, ReportType,
)
from uaef.reporting.regression import RegressionReportGenerator
from uaef.reporting.safety import SafetyReportGenerator
from uaef.reporting.dimension_deepdive import DimensionDashboardGenerator

# Models
from uaef.models.metric_score import MetricScore
from uaef.models.dimension_result import DimensionResult
from uaef.models.evaluation_result import EvaluationResult

print("All imports successful ✓")

---
## 2. Load Evaluation Data & Build Experiment Runs

We reuse the same CSV-to-EvaluationResult conversion from the experiment management notebook.

In [ ]:
from uaef.utils.constants import BATCH_EVAL_OUTPUT_DIR

DATA_DIR = BATCH_EVAL_OUTPUT_DIR

bedrock_df  = pd.read_csv(f"{DATA_DIR}/bedrock_batch_results_20260325_152937.csv")
langgraph_df = pd.read_csv(f"{DATA_DIR}/langgraph_batch_results_20260325_153024.csv")
langchain_df = pd.read_csv(f"{DATA_DIR}/langchain_batch_results_20260325_153126.csv")

# Map CSV column prefixes to UAEF dimension keys (must match DIMENSION_DEFINITIONS)
DIMENSION_METRICS = {
    "multi_agent":      ("dim_Multi-Agent",      ["agent_utilization", "coordination_efficiency", "delegation_quality", "workflow_completion"]),
    "performance":      ("dim_Performance",      ["cost_efficiency", "latency_score", "throughput", "token_efficiency"]),
    "tool_calling":     ("dim_Tool Calling",     ["mcp_compliance", "parameter_quality", "tool_selection_accuracy", "tool_sequence_correctness"]),
    "responsible_ai":   ("dim_Responsible AI",   ["prompt_injection_detection", "bias_score", "safety_score", "toxicity_score"]),
    "multi_turn":       ("dim_Multi-Turn",       ["turn_efficiency", "coherence", "context_retention", "conversation_completeness"]),
    "response_quality": ("dim_Response Quality", ["accuracy", "answer_relevance", "completeness", "hallucination_score"]),
    "reasoning":        ("dim_Reasoning",        ["chain_of_thought_coherence", "fallacy_detection", "logical_consistency", "reasoning_step_correctness"]),
}
DIM_WEIGHT = round(1.0 / len(DIMENSION_METRICS), 4)


def row_to_evaluation(row) -> EvaluationResult:
    dim_results = []
    for dim_key, (csv_col, metric_names) in DIMENSION_METRICS.items():
        scores = [MetricScore(metric_name=m, score=float(row[m]), reasoning="From CSV") for m in metric_names]
        dim_results.append(DimensionResult(
            dimension_name=dim_key, metric_scores=scores,
            aggregate_score=float(row[csv_col]), weight=DIM_WEIGHT,
        ))
    return EvaluationResult(
        trace_id=uuid4(), dimension_results=dim_results,
        overall_score=float(row["overall_score"]), passed=bool(row["passed"]),
        failures=[] if row["passed"] else ["Below threshold"], warnings=[],
        metadata={"test_case": int(row["test_case"]), "query": row["query"]},
    )


# Convert all CSVs
bedrock_evals  = [row_to_evaluation(r) for _, r in bedrock_df.iterrows()]
langgraph_evals = [row_to_evaluation(r) for _, r in langgraph_df.iterrows()]
langchain_evals = [row_to_evaluation(r) for _, r in langchain_df.iterrows()]

print(f"Bedrock evaluations : {len(bedrock_evals)}")
print(f"LangGraph evaluations: {len(langgraph_evals)}")
print(f"LangChain evaluations: {len(langchain_evals)}")

---
## 3. Create Experiment Runs (needed for comparison & regression dashboards)

In [ ]:
manager = ExperimentManager()
experiment = manager.create_experiment(name="QA Agent Framework Comparison", description="Batch eval comparison")

bedrock_run = manager.create_run(
    experiment_id=experiment.experiment_id, run_name="bedrock-baseline",
    agent_design=AgentDesign(framework="bedrock", model="anthropic.claude-3-sonnet-20240229-v1:0"),
    config={"batch_date": "2026-03-25"}, git_commit="a" * 40,
)
for ev in bedrock_evals:
    manager.record_evaluation(bedrock_run.run_id, ev)

langgraph_run = manager.create_run(
    experiment_id=experiment.experiment_id, run_name="langgraph-candidate",
    agent_design=AgentDesign(framework="langgraph", model="anthropic.claude-3-sonnet-20240229-v1:0", tools=["get_weather"]),
    config={"batch_date": "2026-03-25"}, git_commit="b" * 40,
)
for ev in langgraph_evals:
    manager.record_evaluation(langgraph_run.run_id, ev)

langchain_run = manager.create_run(
    experiment_id=experiment.experiment_id, run_name="langchain-candidate",
    agent_design=AgentDesign(framework="langchain", model="anthropic.claude-3-sonnet-20240229-v1:0", tools=["search_knowledge_base"]),
    config={"batch_date": "2026-03-25"}, git_commit="c" * 40,
)
for ev in langchain_evals:
    manager.record_evaluation(langchain_run.run_id, ev)

manager.set_baseline(experiment.experiment_id, bedrock_run.run_id)

print("Runs created and evaluations recorded ✓")
for r in manager.list_runs(experiment.experiment_id):
    print(f"  {r.run_name:25s}  evals={r.evaluation_count}  metrics={len(r.aggregate_metrics)}")

---
## 4. Executive Dashboard

The `ExecutiveDashboardGenerator` produces a high-level overview from evaluation results:
overall score, dimension breakdown, key metrics, pass/fail distribution, critical issues, and trend indicators.

In [ ]:
exec_gen = ExecutiveDashboardGenerator()

exec_dashboard = exec_gen.generate_dashboard(
    results=bedrock_evals,
    title="Executive Dashboard — Bedrock Agent",
    run_id=bedrock_run.run_id,
    experiment_id=experiment.experiment_id,
    include_trends=True,
    include_recommendations=True,
)

print(f"Dashboard: {exec_dashboard.title}")
print(f"  ID         : {exec_dashboard.dashboard_id}")
print(f"  Description: {exec_dashboard.description[:120]}...")
print(f"  Charts     : {len(exec_dashboard.charts)}")
print(f"  Created at : {exec_dashboard.created_at}")
print()
print("Charts:")
for i, chart in enumerate(exec_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
    print(f"     Data keys: {list(chart.data.keys())}")
print()
print("Metadata keys:", list(exec_dashboard.metadata.keys()))

---
## 5. Inspect Chart Data Structures

Each `Chart` object contains structured data that can be rendered by the `VisualizationEngine`
or consumed directly. Let's look at what's inside.

In [ ]:
for chart in exec_dashboard.charts:
    print(f"=== {chart.title} ({chart.chart_type.value}) ===")
    for key, val in chart.data.items():
        if isinstance(val, list) and len(val) > 5:
            print(f"  {key}: [{val[0]}, {val[1]}, ... ] ({len(val)} items)")
        else:
            print(f"  {key}: {val}")
    if chart.description:
        print(f"  description: {chart.description}")
    print()

---
## 6. Comparison Dashboard

The `ComparisonDashboardGenerator` produces side-by-side charts, delta visualizations,
percentage change charts, regression/improvement summaries, statistical significance tests,
and comparison heatmaps.

In [ ]:
comp_gen = ComparisonDashboardGenerator(regression_threshold=5.0, improvement_threshold=5.0)

comp_dashboard = comp_gen.generate_dashboard(
    baseline_run=bedrock_run,
    comparison_run=langgraph_run,
    title="Comparison — Bedrock vs LangGraph",
    include_statistical_tests=True,
)

print(f"Dashboard: {comp_dashboard.title}")
print(f"  Charts: {len(comp_dashboard.charts)}")
print(f"  Description: {comp_dashboard.description[:150]}...")
print()
print("Charts:")
for i, chart in enumerate(comp_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
print()
print(f"Regressions : {comp_dashboard.metadata.get('regression_count')}")
print(f"Improvements: {comp_dashboard.metadata.get('improvement_count')}")
print(f"Unchanged   : {comp_dashboard.metadata.get('unchanged_count')}")

---
## 7. Multi-Run Comparison Dashboard

Compare multiple candidates against the baseline in a single dashboard.

In [ ]:
multi_dashboard = comp_gen.generate_multi_run_comparison(
    baseline_run=bedrock_run,
    comparison_runs=[langgraph_run, langchain_run],
    title="Multi-Run Comparison — All Frameworks",
)

print(f"Dashboard: {multi_dashboard.title}")
print(f"  Charts: {len(multi_dashboard.charts)}")
print()
for i, chart in enumerate(multi_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
    print(f"     Data keys: {list(chart.data.keys())}")

---
## 8. Regression Dashboard

The `RegressionReportGenerator` produces a detailed regression analysis with severity
distribution, before/after comparison, impact heatmap, root cause analysis, and recommendations.

Note: This requires actual regressions to exist between the runs.

In [ ]:
reg_gen = RegressionReportGenerator(regression_threshold=5.0)

# Bedrock → LangGraph has regressions
reg_dashboard = reg_gen.generate_regression_dashboard(
    baseline_run=bedrock_run,
    current_run=langgraph_run,
    title="Regression Report — Bedrock → LangGraph",
    include_root_cause=True,
    include_recommendations=True,
)

print(f"Dashboard: {reg_dashboard.title}")
print(f"  Charts: {len(reg_dashboard.charts)}")
print(f"  Description: {reg_dashboard.description[:150]}...")
print()
print("Charts:")
for i, chart in enumerate(reg_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
print()
print(f"Total regressions: {reg_dashboard.metadata.get('total_regressions')}")
print(f"Critical: {reg_dashboard.metadata.get('critical_count')}")
print(f"High    : {reg_dashboard.metadata.get('high_count')}")
print(f"Medium  : {reg_dashboard.metadata.get('medium_count')}")
print(f"Low     : {reg_dashboard.metadata.get('low_count')}")

if "root_causes" in reg_dashboard.metadata:
    print(f"\nRoot causes identified: {len(reg_dashboard.metadata['root_causes'])}")
    for rc in reg_dashboard.metadata["root_causes"][:3]:
        print(f"  • {rc.get('category', 'N/A')}: {rc.get('description', 'N/A')[:80]}")

if "recommendations" in reg_dashboard.metadata:
    print(f"\nRecommendations: {len(reg_dashboard.metadata['recommendations'])}")
    for rec in reg_dashboard.metadata["recommendations"][:3]:
        print(f"  • [{rec.get('priority', 'N/A')}] {rec.get('title', 'N/A')}")

---
## 9. Historical Regression Dashboard

Track regressions across multiple runs over time — shows a regression timeline,
metric regression heatmap, and severity trends.

In [ ]:
hist_dashboard = reg_gen.generate_historical_regression_dashboard(
    baseline_run=bedrock_run,
    comparison_runs=[langgraph_run, langchain_run],
    title="Historical Regression Tracking",
)

print(f"Dashboard: {hist_dashboard.title}")
print(f"  Charts: {len(hist_dashboard.charts)}")
print(f"  Total regressions across all runs: {hist_dashboard.metadata.get('total_regressions')}")
print()
for i, chart in enumerate(hist_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
    print(f"     Data keys: {list(chart.data.keys())}")

---
## 10. Safety Dashboard

The `SafetyReportGenerator` focuses on responsible AI metrics: safety scores, bias detection,
toxicity breakdown, prompt injection attempts, and safety trends.

In [ ]:
safety_gen = SafetyReportGenerator()

safety_dashboard = safety_gen.generate_safety_dashboard(
    results=bedrock_evals,
    title="Safety Report — Bedrock Agent",
    run_id=bedrock_run.run_id,
    include_recommendations=True,
)

print(f"Dashboard: {safety_dashboard.title}")
print(f"  Charts: {len(safety_dashboard.charts)}")
print(f"  Description: {safety_dashboard.description[:150]}...")
print()
print("Charts:")
for i, chart in enumerate(safety_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
print()
print(f"Safety concerns: {safety_dashboard.metadata.get('safety_concerns_count', 0)}")
safety_metrics = safety_dashboard.metadata.get("safety_metrics", {})
for key in ["safety_score", "bias_score", "toxicity_score"]:
    if key in safety_metrics:
        print(f"  {key}: {safety_metrics[key]}")

---
## 11. Dimension Deep-Dive Dashboard

Drill into a specific dimension (e.g., `tool_calling`, `response_quality`, `reasoning`)
to see metric distributions, correlations, top failures, and trends.

In [ ]:
dim_gen = DimensionDashboardGenerator()

# Deep-dive into response_quality
dim_dashboard = dim_gen.generate_dashboard(
    results=bedrock_evals,
    dimension="response_quality",
    title="Deep Dive — Response Quality (Bedrock)",
    run_id=bedrock_run.run_id,
    include_trends=True,
    include_recommendations=True,
)

print(f"Dashboard: {dim_dashboard.title}")
print(f"  Charts: {len(dim_dashboard.charts)}")
print(f"  Dimension: {dim_dashboard.metadata.get('dimension_name')}")
print()
print("Charts:")
for i, chart in enumerate(dim_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
print()
stats = dim_dashboard.metadata.get("dimension_statistics", {})
print("Dimension statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

---
## 12. Dimension Deep-Dive — Tool Calling

In [ ]:
tool_dashboard = dim_gen.generate_dashboard(
    results=langgraph_evals,
    dimension="tool_calling",
    title="Deep Dive — Tool Calling (LangGraph)",
    run_id=langgraph_run.run_id,
)

print(f"Dashboard: {tool_dashboard.title}")
print(f"  Charts: {len(tool_dashboard.charts)}")
for i, chart in enumerate(tool_dashboard.charts):
    print(f"  {i+1}. [{chart.chart_type.value:10s}] {chart.title}")
print()
stats = tool_dashboard.metadata.get("dimension_statistics", {})
print(f"Average score: {stats.get('average_score', 'N/A')}")
print(f"Pass rate    : {stats.get('pass_rate', 'N/A')}")
print(f"Failed count : {stats.get('failed_count', 'N/A')}")

---
## 13. Wrapping Dashboards in a Report

A `Report` wraps a `Dashboard` with a summary, sections, and export format metadata.
This is the top-level object you'd hand to the exporter.

In [ ]:
exec_report = Report(
    title="Executive Report — Bedrock Agent Q1 2026",
    report_type=ReportType.EXECUTIVE,
    dashboard=exec_dashboard,
    summary=exec_dashboard.description or "Executive performance overview.",
    sections=[
        {
            "title": "Key Findings",
            "content": f"Evaluated {len(bedrock_evals)} test cases. "
                       f"Overall pass rate: {sum(1 for e in bedrock_evals if e.passed)}/{len(bedrock_evals)}."
        },
        {
            "title": "Recommendations",
            "content": "All safety metrics at 1.0. Focus optimization on response completeness."
        },
    ],
    metadata={"experiment_id": str(experiment.experiment_id), "run_id": str(bedrock_run.run_id)},
    export_formats=[ReportFormat.JSON, ReportFormat.HTML, ReportFormat.CSV],
)

print(f"Report: {exec_report.title}")
print(f"  Type    : {exec_report.report_type.value}")
print(f"  Sections: {len(exec_report.sections)}")
print(f"  Formats : {[f.value for f in exec_report.export_formats]}")
print(f"  Summary : {exec_report.summary[:120]}...")

---
## 14. Export to JSON

The `ReportExporter` serializes Reports and Dashboards to JSON, HTML, and CSV.

In [ ]:
exporter = ReportExporter()

# Export report to JSON
json_str = exporter.export_to_json(exec_report)
print(f"JSON export length: {len(json_str)} chars")
print(f"First 300 chars:\n{json_str[:300]}...")

# Also export a bare dashboard
dash_json = exporter.export_to_json(exec_dashboard)
print(f"\nDashboard JSON length: {len(dash_json)} chars")

---
## 15. Export to HTML

Generates a styled HTML page. Uses Jinja2 templates if available, otherwise falls back
to basic HTML generation.

In [ ]:
html_str = exporter.export_to_html(exec_report)
print(f"HTML export length: {len(html_str)} chars")
print(f"Contains <html>: {'<html>' in html_str.lower() or '<div>' in html_str.lower()}")
print(f"First 300 chars:\n{html_str[:300]}...")

---
## 16. Export to CSV

Extracts tabular data from charts and exports as CSV — useful for further analysis in
spreadsheets or BI tools.

In [ ]:
csv_str = exporter.export_to_csv(exec_dashboard)
print(f"CSV export length: {len(csv_str)} chars")
print(f"First 500 chars:\n{csv_str[:500]}...")

---
## 17. Export to Files

All export methods accept an `output_path` parameter to write directly to disk.

In [ ]:
import os

OUTPUT_DIR = os.path.join(os.getcwd(), "..", "output", "reports")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# JSON
exporter.export_to_json(exec_report, output_path=os.path.join(OUTPUT_DIR, "executive_report.json"))
print(f"✓ Wrote {OUTPUT_DIR}/executive_report.json")

# HTML
exporter.export_to_html(exec_report, output_path=os.path.join(OUTPUT_DIR, "executive_report.html"))
print(f"✓ Wrote {OUTPUT_DIR}/executive_report.html")

# CSV
exporter.export_to_csv(exec_dashboard, output_path=os.path.join(OUTPUT_DIR, "executive_dashboard.csv"))
print(f"✓ Wrote {OUTPUT_DIR}/executive_dashboard.csv")

# Export comparison dashboard too
exporter.export_to_json(comp_dashboard, output_path=os.path.join(OUTPUT_DIR, "comparison_dashboard.json"))
print(f"✓ Wrote {OUTPUT_DIR}/comparison_dashboard.json")

exporter.export_to_csv(comp_dashboard, output_path=os.path.join(OUTPUT_DIR, "comparison_dashboard.csv"))
print(f"✓ Wrote {OUTPUT_DIR}/comparison_dashboard.csv")

---
## 18. Visualization Engine

The `VisualizationEngine` renders `Chart` objects into interactive HTML (plotly) or static
PNG (matplotlib). It supports 8 chart types: bar, line, scatter, heatmap, pie, radar, box, histogram.

It also supports theming via `apply_theme()`.

> **Note**: If plotly/matplotlib are not installed, the engine will raise an `ImportError`.
> Install them with `pip install plotly matplotlib` to enable rendering.

In [ ]:
try:
    viz = VisualizationEngine()
    viz_available = True
    print("VisualizationEngine initialized ✓")
except ImportError as e:
    viz_available = False
    print(f"VisualizationEngine not available: {e}")
    print("Install with: pip install plotly matplotlib")

# Check available backends
try:
    import plotly
    print(f"Plotly available: {plotly.__version__}")
except ImportError:
    print("Plotly not installed — interactive charts unavailable")

try:
    import matplotlib
    print(f"Matplotlib available: {matplotlib.__version__}")
except ImportError:
    print("Matplotlib not installed — static charts unavailable")

print(f"\nSupported chart types: {[ct.value for ct in ChartType]}")

---
## 19. Render Charts (if visualization libraries are available)

We'll try to render a few charts from the executive dashboard. If no viz library is
installed, we show the raw chart data instead.

In [ ]:
if not viz_available:
    print("VisualizationEngine not available — showing raw chart data instead.")
    print("Install plotly or matplotlib to enable rendering.\n")
    for chart in exec_dashboard.charts:
        print(f"  {chart.title} ({chart.chart_type.value}): {list(chart.data.keys())}")
else:
    # Pick a bar chart from the executive dashboard
    bar_charts = [c for c in exec_dashboard.charts if c.chart_type == ChartType.BAR]

    if bar_charts:
        chart = bar_charts[0]
        print(f"Rendering: {chart.title} ({chart.chart_type.value})")
        try:
            result = viz.generate_chart(chart, interactive=True)
            print(f"  → Interactive HTML generated ({len(result)} chars)")
        except ImportError:
            result = viz.generate_chart(chart, interactive=False)
            print(f"  → Static PNG generated ({len(result)} bytes)")

    # Try a radar chart
    radar_charts = [c for c in exec_dashboard.charts if c.chart_type == ChartType.RADAR]
    if radar_charts:
        chart = radar_charts[0]
        print(f"\nRendering: {chart.title} ({chart.chart_type.value})")
        try:
            result = viz.generate_chart(chart, interactive=True)
            print(f"  → Interactive HTML generated ({len(result)} chars)")
        except ImportError:
            result = viz.generate_chart(chart, interactive=False)
            print(f"  → Static PNG generated ({len(result)} bytes)")

    # Try a pie chart
    pie_charts = [c for c in exec_dashboard.charts if c.chart_type == ChartType.PIE]
    if pie_charts:
        chart = pie_charts[0]
        print(f"\nRendering: {chart.title} ({chart.chart_type.value})")
        try:
            result = viz.generate_chart(chart, interactive=True)
            print(f"  → Interactive HTML generated ({len(result)} chars)")
        except ImportError:
            result = viz.generate_chart(chart, interactive=False)
            print(f"  → Static PNG generated ({len(result)} bytes)")

---
## 20. Build a Custom Chart

You can create `Chart` objects directly and render them with the `VisualizationEngine`.

In [ ]:
custom_chart = Chart(
    title="Framework Overall Scores",
    chart_type=ChartType.BAR,
    data={
        "labels": ["Bedrock", "LangGraph", "LangChain"],
        "values": [
            sum(e.overall_score for e in bedrock_evals) / len(bedrock_evals),
            sum(e.overall_score for e in langgraph_evals) / len(langgraph_evals),
            sum(e.overall_score for e in langchain_evals) / len(langchain_evals),
        ],
    },
    labels=["Bedrock", "LangGraph", "LangChain"],
    colors=["#4CAF50", "#FF9800", "#2196F3"],
    options={"yaxis_title": "Overall Score"},
    description="Average overall score per framework",
)

print(f"Custom chart: {custom_chart.title}")
print(f"  Type  : {custom_chart.chart_type.value}")
print(f"  Labels: {custom_chart.labels}")
print(f"  Values: {[f'{v:.4f}' for v in custom_chart.data['values']]}")
print(f"  Colors: {custom_chart.colors}")

try:
    html = viz.generate_chart(custom_chart, interactive=True)
    print(f"  → Rendered as interactive HTML ({len(html)} chars)")
except (ImportError, NameError):
    try:
        png = viz.generate_chart(custom_chart, interactive=False)
        print(f"  → Rendered as static PNG ({len(png)} bytes)")
    except (ImportError, NameError):
        print("  → No viz library available for rendering")

---
## 21. Build a Custom Dashboard from Scratch

Compose multiple charts into a `Dashboard` manually.

In [ ]:
pass_counts = {
    "Bedrock": sum(1 for e in bedrock_evals if e.passed),
    "LangGraph": sum(1 for e in langgraph_evals if e.passed),
    "LangChain": sum(1 for e in langchain_evals if e.passed),
}

pass_chart = Chart(
    title="Pass Rate by Framework",
    chart_type=ChartType.BAR,
    data={
        "labels": list(pass_counts.keys()),
        "values": [v / 4 * 100 for v in pass_counts.values()],  # percentage
    },
    colors=["#4CAF50", "#FF9800", "#2196F3"],
    options={"yaxis_title": "Pass Rate (%)"},
)

custom_dashboard = Dashboard(
    title="Custom Framework Comparison Dashboard",
    description="Hand-built dashboard comparing all three frameworks.",
    charts=[custom_chart, pass_chart],
    metadata={"dashboard_type": "custom", "frameworks": ["bedrock", "langgraph", "langchain"]},
)

print(f"Dashboard: {custom_dashboard.title}")
print(f"  Charts: {len(custom_dashboard.charts)}")
for i, c in enumerate(custom_dashboard.charts):
    print(f"  {i+1}. {c.title} ({c.chart_type.value})")

# Export it
json_out = exporter.export_to_json(custom_dashboard)
print(f"\nJSON export: {len(json_out)} chars")
csv_out = exporter.export_to_csv(custom_dashboard)
print(f"CSV export: {len(csv_out)} chars")

---
## 22. Model Validation

The reporting models enforce constraints — empty titles, empty charts, invalid sections, etc.

In [ ]:
from pydantic import ValidationError

# Empty dashboard title
try:
    Dashboard(title="", charts=[custom_chart])
except ValidationError as e:
    print(f"✓ Empty dashboard title rejected")

# Dashboard with no charts
try:
    Dashboard(title="Empty", charts=[])
except ValidationError as e:
    print(f"✓ Dashboard with no charts rejected")

# Empty chart title
try:
    Chart(title="", chart_type=ChartType.BAR, data={"x": [1]})
except ValidationError as e:
    print(f"✓ Empty chart title rejected")

# Chart with empty data
try:
    Chart(title="Test", chart_type=ChartType.BAR, data={})
except ValidationError as e:
    print(f"✓ Chart with empty data rejected")

# Invalid hex color
try:
    Chart(title="Test", chart_type=ChartType.BAR, data={"x": [1]}, colors=["#ZZZZZZ"])
except ValidationError as e:
    print(f"✓ Invalid hex color rejected")

# Report with invalid section (missing content)
try:
    Report(
        title="Bad", report_type=ReportType.EXECUTIVE,
        dashboard=custom_dashboard, summary="Test",
        sections=[{"title": "Missing content"}],
    )
except ValidationError as e:
    print(f"✓ Section without content rejected")

# Report with empty summary
try:
    Report(
        title="Bad", report_type=ReportType.EXECUTIVE,
        dashboard=custom_dashboard, summary="   ",
    )
except ValidationError as e:
    print(f"✓ Empty report summary rejected")

# Export None
try:
    exporter.export_to_json(None)
except ValueError as e:
    print(f"✓ Export None rejected")

print("\nAll validations passed ✓")

---
## 23. Summary

| Feature | Covered |
|---|---|
| `ExecutiveDashboardGenerator` — overall score, dimensions, key metrics, pass/fail, critical issues, trends | ✓ |
| `ComparisonDashboardGenerator` — side-by-side, delta, % change, regression/improvement, heatmap | ✓ |
| `ComparisonDashboardGenerator` — multi-run comparison | ✓ |
| `RegressionReportGenerator` — regression dashboard with severity, root cause, recommendations | ✓ |
| `RegressionReportGenerator` — historical regression tracking | ✓ |
| `SafetyReportGenerator` — safety overview, bias, toxicity, injection, trends | ✓ |
| `DimensionDashboardGenerator` — metric distribution, correlations, failures, trends | ✓ |
| `Chart` model — all 8 chart types (bar, line, scatter, heatmap, pie, radar, box, histogram) | ✓ |
| `Dashboard` model — compose charts with metadata and layout | ✓ |
| `Report` model — wrap dashboard with summary, sections, export formats | ✓ |
| `ReportExporter` — export to JSON | ✓ |
| `ReportExporter` — export to HTML | ✓ |
| `ReportExporter` — export to CSV | ✓ |
| `ReportExporter` — export to files on disk | ✓ |
| `VisualizationEngine` — render charts (plotly/matplotlib) | ✓ |
| Custom chart and dashboard creation | ✓ |
| Model validation (titles, data, colors, sections) | ✓ |